In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from datetime import datetime
from keras_tuner import RandomSearch
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# root_dir = 'trim_data/data_set'
root_dir = 'data_set'
# dataset = 'merged_sliding_window.csv'
# grouping_col = 'window_number'
# model = 'lstm'
dataset = 'merged_gait_segmentation_interpolated.csv'
grouping_col = 'stepcount'
model = 'cnn'

In [3]:
# cols = [
#     'Left_Hallux_raw', 'Right_Hallux_raw',
#     'Left_Toes_raw', 'Right_Toes_raw',
#     'Left_Met1_raw', 'Left_Met3_raw', 'Left_Met5_raw',
#     'Right_Met1_raw', 'Right_Met3_raw', 'Right_Met5_raw', 
#     'Left_Arch_raw', 'Right_Arch_raw',
#     'Left_Heel_R_raw', 'Left_Heel_L_raw', 
#     'Right_Heel_L_raw', 'Right_Heel_R_raw',

#     "acceleration_Pelvis_x_local","acceleration_Pelvis_y_local","acceleration_Pelvis_z_local",
#     "acceleration_RightForeArm_x_local","acceleration_RightForeArm_y_local","acceleration_RightForeArm_z_local",
#     "acceleration_RightUpperLeg_x_local","acceleration_RightUpperLeg_y_local","acceleration_RightUpperLeg_z_local",
#     "acceleration_RightLowerLeg_x_local","acceleration_RightLowerLeg_y_local","acceleration_RightLowerLeg_z_local",
#     "acceleration_RightFoot_x_local","acceleration_RightFoot_y_local","acceleration_RightFoot_z_local",
#     "acceleration_RightToe_x_local","acceleration_RightToe_y_local","acceleration_RightToe_z_local",
#     "acceleration_LeftUpperLeg_x_local","acceleration_LeftUpperLeg_y_local","acceleration_LeftUpperLeg_z_local",
#     "acceleration_LeftLowerLeg_x_local","acceleration_LeftLowerLeg_y_local","acceleration_LeftLowerLeg_z_local",
#     "acceleration_LeftFoot_x_local","acceleration_LeftFoot_y_local","acceleration_LeftFoot_z_local",
#     "acceleration_LeftToe_x_local","acceleration_LeftToe_y_local","acceleration_LeftToe_z_local",

#     "angularVelocity_Pelvis_x_local","angularVelocity_Pelvis_y_local","angularVelocity_Pelvis_z_local",
#     "angularVelocity_RightForeArm_x_local","angularVelocity_RightForeArm_y_local","angularVelocity_RightForeArm_z_local",
#     "angularVelocity_RightUpperLeg_x_local","angularVelocity_RightUpperLeg_y_local","angularVelocity_RightUpperLeg_z_local",
#     "angularVelocity_RightLowerLeg_x_local","angularVelocity_RightLowerLeg_y_local","angularVelocity_RightLowerLeg_z_local",
#     "angularVelocity_RightFoot_x_local","angularVelocity_RightFoot_y_local","angularVelocity_RightFoot_z_local",
#     "angularVelocity_RightToe_x_local","angularVelocity_RightToe_y_local","angularVelocity_RightToe_z_local",
#     "angularVelocity_LeftUpperLeg_x_local","angularVelocity_LeftUpperLeg_y_local","angularVelocity_LeftUpperLeg_z_local",
#     "angularVelocity_LeftLowerLeg_x_local","angularVelocity_LeftLowerLeg_y_local","angularVelocity_LeftLowerLeg_z_local",
#     "angularVelocity_LeftFoot_x_local","angularVelocity_LeftFoot_y_local","angularVelocity_LeftFoot_z_local",
#     "angularVelocity_LeftToe_x_local","angularVelocity_LeftToe_y_local","angularVelocity_LeftToe_z_local",

#     'participant_id',  'walk_mode'
# ]

cols = [
    "acceleration_Pelvis_x_local","acceleration_Pelvis_y_local","acceleration_Pelvis_z_local",
    "acceleration_RightForeArm_x_local","acceleration_RightForeArm_y_local","acceleration_RightForeArm_z_local",
    "acceleration_RightUpperLeg_x_local","acceleration_RightUpperLeg_y_local","acceleration_RightUpperLeg_z_local",
    "acceleration_RightLowerLeg_x_local","acceleration_RightLowerLeg_y_local","acceleration_RightLowerLeg_z_local",
    "acceleration_LeftUpperLeg_x_local","acceleration_LeftUpperLeg_y_local","acceleration_LeftUpperLeg_z_local",
    "acceleration_LeftLowerLeg_x_local","acceleration_LeftLowerLeg_y_local","acceleration_LeftLowerLeg_z_local",

    "angularVelocity_Pelvis_x_local","angularVelocity_Pelvis_y_local","angularVelocity_Pelvis_z_local",
    "angularVelocity_RightForeArm_x_local","angularVelocity_RightForeArm_y_local","angularVelocity_RightForeArm_z_local",
    "angularVelocity_RightUpperLeg_x_local","angularVelocity_RightUpperLeg_y_local","angularVelocity_RightUpperLeg_z_local",
    "angularVelocity_RightLowerLeg_x_local","angularVelocity_RightLowerLeg_y_local","angularVelocity_RightLowerLeg_z_local",
    "angularVelocity_LeftUpperLeg_x_local","angularVelocity_LeftUpperLeg_y_local","angularVelocity_LeftUpperLeg_z_local",
    "angularVelocity_LeftLowerLeg_x_local","angularVelocity_LeftLowerLeg_y_local","angularVelocity_LeftLowerLeg_z_local",

    'participant_id',  'walk_mode'
]
cols.append(grouping_col)

In [ ]:
def compute_metrics(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    tn = []
    fp = []
    fn = []
    tp = []
    
    for i in range(len(labels)):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)
        tn.append(TN)
        fp.append(FP)
        fn.append(FN)
        tp.append(TP)

    sensitivity = np.mean([tp[i] / (tp[i] + fn[i]) if (tp[i] + fn[i]) > 0 else 0 for i in range(len(labels))])
    specificity = np.mean([tn[i] / (tn[i] + fp[i]) if (tn[i] + fp[i]) > 0 else 0 for i in range(len(labels))])
    
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    
    return acc, f1, sensitivity, specificity

In [5]:
def plot_normalized_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix (Normalized)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

In [ ]:
df = pd.read_csv(os.path.join(root_dir, dataset))
df = df[cols]
le = LabelEncoder()
df['walk_mode_enc'] = le.fit_transform(df['walk_mode'])

# Subject-wise split
participant_ids = df['participant_id'].unique()
train_ids, test_ids = train_test_split(participant_ids, test_size=0.2, random_state=42)

train_df = df[df['participant_id'].isin(train_ids)]
test_df = df[df['participant_id'].isin(test_ids)]

# Feature columns
exclude_cols = ['participant_id', 'walk_mode', 'walk_mode_enc', grouping_col]
input_cols = [col for col in df.columns if col not in exclude_cols]

# Scaling
scaler = StandardScaler()
train_df[input_cols] = scaler.fit_transform(train_df[input_cols])
test_df[input_cols] = scaler.transform(test_df[input_cols])

def reshape(df, group_col=grouping_col):
    grouped = df.groupby(group_col)
    X = np.stack([group[input_cols].values for _, group in grouped])
    y = np.array([group['walk_mode_enc'].iloc[0] for _, group in grouped])
    return X, y

X_train, y_train = reshape(train_df)
X_test, y_test = reshape(test_df)

num_classes = len(le.classes_)
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

class_weights_array = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(zip(np.unique(y_train), class_weights_array))
total = sum(class_weights.values())
for k in class_weights:
    class_weights[k] = total / (len(class_weights) * class_weights[k])

# CNN Model Builder
def build_cnn(hp):
    model = models.Sequential()
    model.add(layers.Conv1D(filters=hp.Int('filters', 32, 128, step=32), kernel_size=hp.Choice('kernel_size', values=[3, 5, 7]), activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Conv1D(filters=hp.Int('filters2', 32, 128, step=32), kernel_size=hp.Choice('kernel_size2', values=[3, 5]), activation='relu'))
    model.add(layers.GlobalAveragePooling1D())
    model.add(layers.Dropout(hp.Float('dropout', 0.2, 0.5)))
    model.add(layers.Dense(hp.Int('dense_units', 32, 128, step=32), activation='relu'))
    model.add(layers.Dense(num_classes, activation='softmax'))
    learning_rate = hp.Choice('learning_rate', values=[1e-4, 5e-4, 1e-3])
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    model.summary()
    return model

# LSTM Model Builder
def build_lstm(hp):
    model = models.Sequential()
    model.add(layers.LSTM(hp.Int('lstm_units', 32, 128, step=32), input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
    model.add(layers.Dropout(hp.Float('dropout1', 0.2, 0.5)))
    model.add(layers.LSTM(hp.Int('lstm_units_2', 32, 128, step=32)))
    model.add(layers.Dropout(hp.Float('dropout2', 0.2, 0.5)))
    model.add(layers.Dense(num_classes, activation='softmax'))
    learning_rate = hp.Choice('learning_rate', values=[1e-4, 5e-4, 1e-3])
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    model.summary()
    return model

if model == 'cnn':
    tuner = RandomSearch(build_cnn, objective='val_accuracy', max_trials=5, directory='cnn_tuner', project_name='honda_project')
elif model == 'lstm':
    tuner = RandomSearch(build_lstm, objective='val_accuracy', max_trials=5, directory='lstm_tuner', project_name='honda_project')

tuner.search(X_train, y_train_cat, epochs=10, validation_split=0.2, class_weight=class_weights)

model = tuner.get_best_models(1)[0]
log_dir = os.path.join("logs", datetime.now().strftime("%Y%m%d-%H%M%S"))
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    TensorBoard(log_dir=log_dir, histogram_freq=1)
]
history = model.fit(X_train, y_train_cat, validation_data=(X_test, y_test_cat), epochs=50, class_weight=class_weights, callbacks=callbacks, verbose=1)

y_pred = np.argmax(model.predict(X_test), axis=1)
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))
labels = np.unique(y_test)
acc, f1, sensitivity, specificity = compute_metrics(y_test, y_pred, labels)

print(f"Global Accuracy     : {acc:.4f}")
print(f"Global F1 Score     : {f1:.4f}")
print(f"Global Sensitivity  : {sensitivity:.4f}")
print(f"Global Specificity  : {specificity:.4f}")

plot_normalized_confusion_matrix(y_test, y_pred, le.classes_)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs
# go to http://localhost:6006/